In [2]:
!pip install nltk scikit-learn numpy

In [3]:
import re
import nltk
import numpy as np

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
class PreprocessingModule:

    def __init__(self):
        self.stop_words = set(stopwords.words('english'))

    def transform(self, text):

        # Empty input
        if not text.strip():
            return ""

        # Lowercase
        text = text.lower()

        # Remove punctuation
        text = re.sub(r'[^\w\s]', '', text)

        # Tokenization
        words = word_tokenize(text)

        # Stopword removal
        filtered_words = [
            word for word in words
            if word not in self.stop_words
        ]

        return " ".join(filtered_words)

In [6]:
class VectorizerModule:

    def __init__(self):
        self.vectorizer = TfidfVectorizer()

    def fit(self, corpus):
        self.corpus_matrix = self.vectorizer.fit_transform(corpus)

    def transform(self, query):
        return self.vectorizer.transform([query])

In [8]:
documents = [

    "Artificial intelligence is transforming healthcare systems",

    "Machine learning improves recommendation engines",

    "Deep learning is widely used in computer vision",

    "Natural language processing powers chatbots",

    "Python is popular for AI development",

    "Cybersecurity protects digital infrastructure",

    "Cloud computing enables scalable applications",

    "Neural networks learn complex patterns",

    "Data preprocessing improves model performance",

    "AI tools help students learn faster",

    "Robotics combines AI and mechanical systems",

    "Big data analytics supports business decisions",

    "Blockchain technology improves transaction security",

    "Autonomous vehicles use machine learning",

    "Computer vision helps facial recognition systems",

    "Virtual assistants use speech recognition",

    "Databases store structured information",

    "Software engineering focuses on system design",

    "Web development uses frontend and backend technologies",

    "Mobile applications improve user accessibility"
]

In [9]:
preprocessor = PreprocessingModule()

processed_docs = []

for doc in documents:
    cleaned_doc = preprocessor.transform(doc)
    processed_docs.append(cleaned_doc)

print(processed_docs)

['artificial intelligence transforming healthcare systems', 'machine learning improves recommendation engines', 'deep learning widely used computer vision', 'natural language processing powers chatbots', 'python popular ai development', 'cybersecurity protects digital infrastructure', 'cloud computing enables scalable applications', 'neural networks learn complex patterns', 'data preprocessing improves model performance', 'ai tools help students learn faster', 'robotics combines ai mechanical systems', 'big data analytics supports business decisions', 'blockchain technology improves transaction security', 'autonomous vehicles use machine learning', 'computer vision helps facial recognition systems', 'virtual assistants use speech recognition', 'databases store structured information', 'software engineering focuses system design', 'web development uses frontend backend technologies', 'mobile applications improve user accessibility']


In [10]:
vectorizer = VectorizerModule()

vectorizer.fit(processed_docs)

corpus_matrix = vectorizer.corpus_matrix

print("TF-IDF matrix created successfully!")

TF-IDF matrix created successfully!


In [11]:
def retrieve(query, corpus_matrix, top_k=3):

    processed_query = preprocessor.transform(query)

    # Handle empty query
    if processed_query == "":
        return "No relevant document found"

    # Convert query to vector
    query_vector = vectorizer.transform(processed_query)

    # Compute similarity
    similarities = cosine_similarity(
        query_vector,
        corpus_matrix
    )[0]

    # Highest score
    max_score = np.max(similarities)

    # Relevance threshold
    if max_score < 0.1:
        return "No relevant document found"

    # Get top results
    top_indices = similarities.argsort()[::-1][:top_k]

    results = []

    for idx in top_indices:

        results.append({
            "document": documents[idx],
            "score": round(float(similarities[idx]), 4)
        })

    return results

In [12]:
queries = [

    "AI in healthcare",

    "machine learning systems",

    "student learning tools",

    "speech recognition assistant",

    "secure digital transactions",

    "frontend website design",

    "car automation using AI",

    # ambiguous queries
    "learning",

    "systems",

    # out-of-domain queries
    "pizza recipe"
]

for query in queries:

    print("=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    results = retrieve(query, corpus_matrix)

    print(results)
    print()

QUERY: AI in healthcare
[{'document': 'Artificial intelligence is transforming healthcare systems', 'score': 0.3641}, {'document': 'Python is popular for AI development', 'score': 0.2672}, {'document': 'Robotics combines AI and mechanical systems', 'score': 0.2389}]

QUERY: machine learning systems
[{'document': 'Machine learning improves recommendation engines', 'score': 0.4899}, {'document': 'Autonomous vehicles use machine learning', 'score': 0.4814}, {'document': 'Robotics combines AI and mechanical systems', 'score': 0.2139}]

QUERY: student learning tools
[{'document': 'AI tools help students learn faster', 'score': 0.3371}, {'document': 'Machine learning improves recommendation engines', 'score': 0.2455}, {'document': 'Autonomous vehicles use machine learning', 'score': 0.2412}]

QUERY: speech recognition assistant
[{'document': 'Virtual assistants use speech recognition', 'score': 0.6245}, {'document': 'Computer vision helps facial recognition systems', 'score': 0.2609}, {'docu

In [13]:
edge_cases = [
    "",
    "a",
    "@@@@1234"
]

for case in edge_cases:

    print("=" * 50)
    print("INPUT:", repr(case))

    result = retrieve(case, corpus_matrix)

    print("OUTPUT:", result)
    print()

INPUT: ''
OUTPUT: No relevant document found

INPUT: 'a'
OUTPUT: No relevant document found

INPUT: '@@@@1234'
OUTPUT: No relevant document found



In [14]:
print("""

                RAW QUERY
                    ↓
         PREPROCESSING MODULE
                    ↓
             CLEANED TEXT
                    ↓
            TF-IDF VECTORIZER
                    ↓
             QUERY VECTOR
                    ↓
           COSINE SIMILARITY
                    ↓
          RANKED DOCUMENTS

""")



                RAW QUERY
                    ↓
         PREPROCESSING MODULE
                    ↓
             CLEANED TEXT
                    ↓
            TF-IDF VECTORIZER
                    ↓
             QUERY VECTOR
                    ↓
           COSINE SIMILARITY
                    ↓
          RANKED DOCUMENTS




In [15]:
failure_analysis = {

    "learning":
    "The query is too ambiguous and matches multiple documents weakly.",

    "systems":
    "The word systems appears in multiple unrelated contexts.",

    "pizza recipe":
    "The knowledge base contains no food-related documents.",

    "car automation using AI":
    "TF-IDF struggles with synonym mismatch between car and autonomous vehicles."
}

for query, reason in failure_analysis.items():

    print("=" * 60)
    print("QUERY:", query)
    print("DIAGNOSIS:", reason)
    print()

QUERY: learning
DIAGNOSIS: The query is too ambiguous and matches multiple documents weakly.

QUERY: systems
DIAGNOSIS: The word systems appears in multiple unrelated contexts.

QUERY: pizza recipe
DIAGNOSIS: The knowledge base contains no food-related documents.

QUERY: car automation using AI
DIAGNOSIS: TF-IDF struggles with synonym mismatch between car and autonomous vehicles.



In [16]:
print("""

TF-IDF retrieval relies heavily on exact vocabulary overlap.

Example:
- Query: car automation
- Document: autonomous vehicles

Even though both have similar meanings,
TF-IDF may fail because the words differ.

This limitation shows why embeddings are important.

Embeddings capture semantic meaning instead of relying only on exact words.

""")



TF-IDF retrieval relies heavily on exact vocabulary overlap.

Example:
- Query: car automation
- Document: autonomous vehicles

Even though both have similar meanings,
TF-IDF may fail because the words differ.

This limitation shows why embeddings are important.

Embeddings capture semantic meaning instead of relying only on exact words.


